# Regression Metrics

**Topic:** Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import FloatSlider, Dropdown, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tkh_utils import PALETTE, FONT, base_layout

housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Small toy dataset used to build intuition in every widget below
np.random.seed(7)
X_synth = np.linspace(1, 9, 9)
Y_synth = 1.4 * X_synth + 2.0 + np.random.normal(0, 1.8, size=9)

mean_y = float(Y_synth.mean())
_ols = LinearRegression().fit(X_synth.reshape(-1, 1), Y_synth)
slope_best = float(_ols.coef_[0])
intercept_best = float(_ols.intercept_)

X_LINE = np.array([X_synth.min() - 0.5, X_synth.max() + 0.5])
R_RANGE = np.linspace(-4, 4, 200)
HILITE_IDX = 5  # index of the point students will drag in the MSE/MAE widgets

---
## What you'll explore

By the end of this session you will be able to:

- **Interpret** R² by comparing a candidate line to the mean baseline, and explain what it means for a line to "beat" that baseline
- **Explain** why squaring residuals in MSE amplifies large errors, and how RMSE brings that penalty back into the original units
- **Describe** how MAE treats every error proportionally, and when that makes it the better choice over MSE

> **Tip:** Start with the R² slider — drag it from 0 to 1 and watch the dashed residual lines shrink. Then try the matching residual-offset slider in both the MSE and MAE widgets and compare how fast each curve climbs.

---
## How we got here

In `ml_concepts/07_loss_functions.ipynb` you saw how loss functions guide the training process, including a side-by-side look at MSE and MAE as loss shapes. In `math/calculus/08_the_cost_function.ipynb` you explored the mathematics behind minimizing prediction error, and `supervised/01_linear_regression.ipynb` showed you the residual diagnostics that come out of a fitted line. This notebook is the natural next step: once training is done, how do you *report* prediction quality in a way that is meaningful to stakeholders — not just minimized during fitting?

The metrics here are what end up in model cards, dashboards, and deployment decisions.

---
## Why this matters for data science

Regression metrics translate abstract "error" into language that connects to real-world impact. An RMSE of 0.4 on California Housing means predictions are off by $40,000 on average (target is in $100k units). Whether that is acceptable depends entirely on the use case — acceptable for a property tax estimate, catastrophic for a mortgage risk model.

Understanding the math behind each metric also reveals its blind spots. A model with low MAE might still have a few massive errors that RMSE would expose. R² can be high even when the data has high natural variance, or misleadingly low when the data barely varies at all.

---
## Try it yourself

### Does the line fit better than just guessing the mean?

Imagine you didn't build a model at all — you just guessed the *average* value every time. That flat average is your baseline. A regression line only earns its keep if it fits the data meaningfully better than that flat guess.

Drag the slider below from **0** (the flat mean) toward **1** (the best-fit line) and watch two things: how much shorter the dashed residual lines get, and how R² climbs from 0 toward its maximum.

In [2]:
out_r2 = Output()

fit_slider = FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.05,
    description="Line fit: mean → best fit",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="460px"),
)

def render_r2(change=None):
    t = fit_slider.value
    slope_t = t * slope_best
    intercept_t = (1 - t) * mean_y + t * intercept_best
    y_line_pts = slope_t * X_synth + intercept_t

    ss_res = float(np.sum((Y_synth - y_line_pts) ** 2))
    ss_tot = float(np.sum((Y_synth - mean_y) ** 2))
    r2 = 1 - ss_res / ss_tot

    traces = []

    for xi, yi, yhat in zip(X_synth, Y_synth, y_line_pts):
        traces.append(go.Scatter(
            x=[xi, xi], y=[yi, yhat],
            mode="lines",
            line=dict(color=PALETTE["muted"], width=1, dash="dot"),
            showlegend=False, hoverinfo="skip",
        ))

    traces.append(go.Scatter(
        x=X_LINE, y=[mean_y, mean_y],
        mode="lines",
        line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
        name="Mean of Y",
    ))
    traces.append(go.Scatter(
        x=X_LINE, y=slope_t * X_LINE + intercept_t,
        mode="lines",
        line=dict(color=PALETTE["primary"], width=3),
        name="Current line",
    ))
    traces.append(go.Scatter(
        x=X_synth, y=Y_synth,
        mode="markers",
        marker=dict(color=PALETTE["accent"], size=10),
        name="Data",
    ))

    layout = base_layout(
        title=f"R² = {r2:.3f}  —  the line explains {max(r2, 0) * 100:.1f}% of the variation the mean alone couldn't",
        xaxis_title="X", yaxis_title="Y",
    )
    layout.update(showlegend=True)

    with out_r2:
        clear_output(wait=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))

fit_slider.observe(render_r2, names="value")
display(VBox([fit_slider, out_r2]))
render_r2()

Formally, R² compares the leftover error of your line (SS_res) to the leftover error of the mean (SS_tot):

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

An R² of 0 means the line is no better than the mean. An R² of 1 means the line passes through every point exactly. R² can even go negative if a line fits *worse* than just guessing the mean.

### What is MSE, really?

Every prediction leaves behind a residual: the gap between the actual value (Y) and what the line predicted (Y′). MSE squares each of those gaps and averages them — and "squaring" is not just algebra. Geometrically, each squared residual is a literal **square** drawn on the gap, and MSE is the **average area** of those squares.

The data below is fixed. What moves is the **line**. Drag the slider and watch two things happen together: the orange squares grow and shrink as the line rotates, and the dot on the right slides along a bowl-shaped curve of MSE values. The line that makes the squares as small as possible — the bottom of the bowl — is exactly the least-squares line that `LinearRegression` finds for you.


In [ ]:
out_mse = Output()

slope_slider = FloatSlider(
    value=slope_best - 0.7, min=slope_best - 0.7, max=slope_best + 0.7, step=0.02,
    description="Slope of the line:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="460px"),
    readout_format=".2f",
)

_mean_x = float(X_synth.mean())
_SLOPE_GRID = np.linspace(slope_best - 0.7, slope_best + 0.7, 120)
_MSE_CURVE = np.array([
    float(np.mean((Y_synth - (m * (X_synth - _mean_x) + mean_y)) ** 2))
    for m in _SLOPE_GRID
])
_MSE_MIN = float(np.mean((Y_synth - (slope_best * (X_synth - _mean_x) + mean_y)) ** 2))

def render_mse(change=None):
    slope_t = slope_slider.value
    y_hat = slope_t * (X_synth - _mean_x) + mean_y
    residuals = Y_synth - y_hat
    mse = float(np.mean(residuals ** 2))
    rmse = float(np.sqrt(mse))
    total_area = mse * len(X_synth)

    fig = make_subplots(
        rows=1, cols=2, column_widths=[0.5, 0.5],
        subplot_titles=(
            "Each residual becomes a square (area = residual²)",
            "MSE as the line's slope changes",
        ),
    )

    # Left panel: literal squares drawn on every residual
    shapes = []
    for xi, yi, yhi, ri in zip(X_synth, Y_synth, y_hat, residuals):
        side = abs(ri)
        if side < 1e-9:
            continue
        # extend each square horizontally toward the center of the plot
        if xi <= _mean_x:
            x0, x1 = xi, xi + side
        else:
            x0, x1 = xi - side, xi
        shapes.append(dict(
            type="rect", xref="x", yref="y",
            x0=x0, x1=x1, y0=min(yi, yhi), y1=max(yi, yhi),
            fillcolor=PALETTE["secondary"], opacity=0.35,
            line=dict(color=PALETTE["secondary"], width=1),
            layer="below",
        ))

    fig.add_trace(go.Scatter(
        x=X_LINE, y=slope_t * (X_LINE - _mean_x) + mean_y,
        mode="lines", line=dict(color=PALETTE["primary"], width=3),
        name="Current line",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=X_synth, y=Y_synth, mode="markers",
        marker=dict(color=PALETTE["accent"], size=10,
                    line=dict(color=PALETTE["primary"], width=1)),
        name="Data",
    ), row=1, col=1)

    # Right panel: the MSE cost curve with the current position and the minimum
    fig.add_trace(go.Scatter(
        x=_SLOPE_GRID, y=_MSE_CURVE, mode="lines",
        line=dict(color=PALETTE["muted"], width=2),
        name="MSE curve", showlegend=False,
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=[slope_best], y=[_MSE_MIN], mode="markers+text",
        marker=dict(color=PALETTE["primary"], size=11, symbol="star"),
        text=["minimum"], textposition="bottom center",
        name="Best-fit slope",
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=[slope_t], y=[mse], mode="markers",
        marker=dict(color=PALETTE["secondary"], size=13),
        name="Current slope",
    ), row=1, col=2)

    fig.update_xaxes(title_text="X", range=[-0.5, 10.5], constrain="domain",
                     gridcolor="#DEE2E6", row=1, col=1)
    fig.update_yaxes(title_text="Y", range=[1.0, 18.0], constrain="domain",
                     scaleanchor="x", scaleratio=1,
                     gridcolor="#DEE2E6", row=1, col=1)
    fig.update_xaxes(title_text="Slope of the line", gridcolor="#DEE2E6", row=1, col=2)
    fig.update_yaxes(title_text="MSE", gridcolor="#DEE2E6", row=1, col=2)

    if mse - _MSE_MIN < 0.05:
        caption = "This is the least-squares line: no other slope makes the squares smaller"
    else:
        caption = f"Total orange area = {total_area:.1f}. Slide toward the star and watch it shrink"

    fig.update_layout(
        title=dict(
            text=f"MSE = {mse:.3f}   |   RMSE = √MSE = {rmse:.3f}<br><sup>{caption}</sup>",
            font=dict(size=FONT["size_title"], family=FONT["family"]),
        ),
        shapes=shapes,
        font=dict(family=FONT["family"]),
        paper_bgcolor=PALETTE["background"],
        plot_bgcolor=PALETTE["surface"],
        showlegend=True,
        height=520,
        margin=dict(l=60, r=30, t=100, b=60),
    )

    with out_mse:
        clear_output(wait=True)
        fig.show()

slope_slider.observe(render_mse, names="value")
display(VBox([slope_slider, out_mse]))
render_mse()


$$MSE = \frac{1}{n}\sum (y_i - \hat{y}_i)^2 \qquad RMSE = \sqrt{MSE}$$

Squaring is why MSE reacts so strongly to outliers: a residual of 4 contributes 16 to the average, while a residual of 2 contributes only 4 — four times smaller, not two. RMSE is simply the square root of that number, brought back into the original units of Y so it is easier to interpret directly.

### The same idea, without the amplification

MAE asks the same question as MSE but takes the absolute value of each residual instead of squaring it. Geometrically, each absolute residual is just the **length** of the vertical bar between the point and the line, and MAE is the **average bar length**. Same setup as before: the data is fixed, the slider rotates the line.

Watch two things: (a) the cost curve on the right is made of straight segments with kinks, not a smooth bowl, and (b) the slope that minimizes MAE is **not** the same as the least-squares slope — the dashed reference line marks where MSE's minimum was. Because MAE never squares, distant points can't shout louder than nearby ones, so its favorite line lands somewhere slightly different.

In [ ]:
out_mae = Output()

slope_slider_mae = FloatSlider(
    value=slope_best - 0.7, min=slope_best - 0.7, max=slope_best + 0.7, step=0.02,
    description="Slope of the line:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="460px"),
    readout_format=".2f",
)

_mean_x_mae = float(X_synth.mean())
_SLOPE_GRID_MAE = np.linspace(slope_best - 0.7, slope_best + 0.7, 400)
_MAE_CURVE = np.array([
    float(np.mean(np.abs(Y_synth - (m * (X_synth - _mean_x_mae) + mean_y))))
    for m in _SLOPE_GRID_MAE
])
_best_idx_mae = int(np.argmin(_MAE_CURVE))
_M_MAE_BEST = float(_SLOPE_GRID_MAE[_best_idx_mae])
_MAE_MIN = float(_MAE_CURVE[_best_idx_mae])

def render_mae(change=None):
    slope_t = slope_slider_mae.value
    y_hat = slope_t * (X_synth - _mean_x_mae) + mean_y
    residuals = Y_synth - y_hat
    mae = float(np.mean(np.abs(residuals)))
    total_length = mae * len(X_synth)

    fig = make_subplots(
        rows=1, cols=2, column_widths=[0.5, 0.5],
        subplot_titles=(
            "Each residual is a bar (length = |residual|)",
            "MAE as the line's slope changes",
        ),
    )

    # Left panel: current line and data
    fig.add_trace(go.Scatter(
        x=X_LINE, y=slope_t * (X_LINE - _mean_x_mae) + mean_y,
        mode="lines", line=dict(color=PALETTE["primary"], width=3),
        name="Current line",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=X_synth, y=Y_synth, mode="markers",
        marker=dict(color=PALETTE["accent"], size=10,
                    line=dict(color=PALETTE["primary"], width=1)),
        name="Data",
    ), row=1, col=1)

    # Left panel: each residual drawn as a thick vertical bar (length = |residual|)
    for i, (xi, yi, yhi) in enumerate(zip(X_synth, Y_synth, y_hat)):
        fig.add_trace(go.Scatter(
            x=[xi, xi], y=[yhi, yi],
            mode="lines",
            line=dict(color=PALETTE["secondary"], width=8),
            opacity=0.6,
            name="Residual bars",
            showlegend=(i == 0),
            hoverinfo="skip",
        ), row=1, col=1)

    # Right panel: the MAE cost curve with the least-squares slope, the MAE minimum, and the current position
    fig.add_trace(go.Scatter(
        x=_SLOPE_GRID_MAE, y=_MAE_CURVE, mode="lines",
        line=dict(color=PALETTE["muted"], width=2),
        name="MAE curve", showlegend=False,
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=[slope_best, slope_best], y=[float(_MAE_CURVE.min()), float(_MAE_CURVE.max())],
        mode="lines",
        line=dict(color=PALETTE["primary"], width=2, dash="dot"),
        name="Least-squares slope",
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=[_M_MAE_BEST], y=[_MAE_MIN], mode="markers+text",
        marker=dict(color=PALETTE["primary"], size=11, symbol="star"),
        text=["MAE minimum"], textposition="bottom center",
        name="MAE-minimizing slope",
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=[slope_t], y=[mae], mode="markers",
        marker=dict(color=PALETTE["secondary"], size=13),
        name="Current slope",
    ), row=1, col=2)

    fig.update_xaxes(title_text="X", range=[-0.5, 10.5],
                     gridcolor="#DEE2E6", row=1, col=1)
    fig.update_yaxes(title_text="Y", range=[1.0, 18.0],
                     gridcolor="#DEE2E6", row=1, col=1)
    fig.update_xaxes(title_text="Slope of the line", gridcolor="#DEE2E6", row=1, col=2)
    fig.update_yaxes(title_text="MAE", gridcolor="#DEE2E6", row=1, col=2)

    if mae - _MAE_MIN < 0.02:
        caption = "This slope minimizes MAE — notice it is not the least-squares slope (dotted line)"
    else:
        caption = f"Total bar length = {total_length:.1f}. Slide toward the star and watch the bars shorten"

    fig.update_layout(
        title=dict(
            text=f"MAE = {mae:.3f}<br><sup>{caption}</sup>",
            font=dict(size=FONT["size_title"], family=FONT["family"]),
        ),
        font=dict(family=FONT["family"]),
        paper_bgcolor=PALETTE["background"],
        plot_bgcolor=PALETTE["surface"],
        showlegend=True,
        height=520,
        margin=dict(l=60, r=30, t=100, b=60),
    )

    with out_mae:
        clear_output(wait=True)
        fig.show()

slope_slider_mae.observe(render_mae, names="value")
display(VBox([slope_slider_mae, out_mae]))
render_mae()

$$MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|$$

Because MAE grows in a straight line rather than a curve, one wildly wrong prediction moves it far less than it moves MSE or RMSE. That is why MAE is considered more robust to outliers — but it also means MAE won't warn you as loudly when your model occasionally makes a very costly mistake.

---
## What's happening?

All four regression metrics measure prediction error, but they define "error" differently.

**R² (Coefficient of Determination)** compares your model's leftover error to the error of the simplest possible baseline — guessing the mean every time. An R² of 0.80 means your line explains 80% of the variation that guessing the mean alone couldn't. It ranges from 0 (no better than the mean) up to 1 (perfect), and can go negative if your model is worse than just guessing the mean.

**MSE (Mean Squared Error)** squares each residual before averaging. Squaring means a $20k error contributes *four times* as much as a $10k error, not just twice. This makes MSE sensitive to large mistakes, and because it is differentiable everywhere, it is the default loss function during training.

**RMSE (Root Mean Squared Error)** is the square root of MSE, brought back into the original units so it is directly interpretable. It shares MSE's sensitivity to large errors.

**MAE (Mean Absolute Error)** takes the average of absolute residuals. It treats all errors proportionally — a $10k error counts the same whether it happens on a $100k or $900k house. Easy to interpret; robust to outliers.

| Metric | Formula | Range | Sensitive to outliers | When to use it |
|---|---|---|---|---|
| R² | 1 − SS_res/SS_tot | (−∞, 1] | Moderate | Comparing models across datasets |
| MSE | mean((y − ŷ)²) | [0, ∞) | Yes | Training loss; not usually reported directly |
| RMSE | √MSE | [0, ∞) | Yes | When large errors matter more than small ones |
| MAE | mean(\|y − ŷ\|) | [0, ∞) | No | When large errors are not especially costly |

---
## Real-world example: Predicting California house prices — how far off is the model?

California Housing is a classic regression benchmark: 20,640 census tracts, 8 features, target is median house value in units of $100k. A Random Forest trained on 80% of the data produces predictions we can evaluate with all four metrics you just explored above.

In [ ]:
rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

abs_err = np.abs(y_test.values - y_pred)
pmin, pmax = float(y_test.min()), float(y_test.max())

layout = base_layout(
    title=f"Random Forest — Predicted vs Actual  (R²={r2:.3f}, MSE={mse:.3f}, RMSE={rmse:.3f}, MAE={mae:.3f})",
    xaxis_title="Actual Median House Value ($100k)",
    yaxis_title="Predicted Median House Value ($100k)",
)
layout.update(
    showlegend=True,
    legend=dict(orientation="h", yanchor="top", y=-0.18, x=0),
    margin=dict(l=60, r=30, t=90, b=60),
)

fig = go.Figure(layout=layout)
fig.add_trace(go.Scatter(
    x=[pmin, pmax], y=[pmin, pmax],
    mode="lines",
    line=dict(color=PALETTE["muted"], width=1.5, dash="dash"),
    name="Perfect prediction",
))
fig.add_trace(go.Scatter(
    x=y_test.values, y=y_pred,
    mode="markers",
    marker=dict(
        color=abs_err,
        colorscale=[[0, PALETTE["accent"]], [0.5, PALETTE["primary"]], [1, PALETTE["secondary"]]],
        size=4,
        opacity=0.6,
        colorbar=dict(title="Abs error", x=1.02),
    ),
    name="Test predictions",
))
fig.show()

#### Reading the metrics

Plugging real numbers into all four metrics: **R² = 0.804**, **MSE = 0.256**, **RMSE = 0.506**, **MAE = 0.330**.

- **R²** — the forest explains about 80% of the variation in house prices that guessing the mean alone could not explain, echoing the mean-baseline widget above.
- **RMSE** — the target is in units of $100k, so an RMSE of 0.506 means typical predictions are off by roughly **$51k**, in the same units as the target itself.
- **MSE** — carries the same information as RMSE, but in squared units, which is why it is hard to interpret directly on its own. It is mainly useful for optimization — recall from the squares widget that MSE is the average area of those squares.
- **MAE vs. RMSE** — MAE (0.330) is noticeably smaller than RMSE (0.506). That gap is the same outlier amplification the MSE and MAE widgets demonstrated, now showing up in real data: a handful of large misses inflate RMSE more than MAE.

Whether roughly $51k of typical error is acceptable depends entirely on the use case, not on the metric.

#### Why is there a vertical stripe at Actual = 5?

The California Housing dataset caps its target at $500,000 — every census tract actually worth more than that is recorded as exactly 5.0 (in $100k units). That is a censoring artifact from how the data was originally collected, not a real property of the housing market.

The model has no such cap, so for these capped tracts it still produces a spread of predictions rather than one fixed value. Those predictions land at whatever the model believes the tract is worth, which is why the capped points smear vertically at the right edge of the plot instead of sitting on the diagonal.

Those same capped points produce some of the largest residuals in the whole plot. Because MSE and RMSE square residuals, this small handful of points contributes disproportionately to both — a real-world instance of the exact outlier sensitivity the MSE and MAE widgets demonstrated above.

This is a data-collection artifact, not a modeling error. In practice, you would decide whether to drop or separately model the capped rows before trusting squared-error metrics on this dataset.

---
## Key takeaway

> **MAE gives you the average mistake in your units; MSE — and its square root, RMSE — punishes large errors harder; R² tells you how much of the story your model explains beyond just guessing the mean; use all four together, never just one.**

---
*Next up: `02_classification_metrics.ipynb` — precision, recall, F1, and the threshold decision that is really a business decision*